# inplace-op-unsafe-warning — worked example 3: Custom impl warns, torch-style raises

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `inplace-op-unsafe-warning`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

PyTorch raises a hard `RuntimeError` when an in-place op would invalidate a value needed for backward. A teaching framework can instead just emit a warning and proceed — trading safety for visibility. Either way the signal is the same: `x.recipe is not None` means the tensor is a graph intermediate and mutating it is unsafe.

## Worked solution

We implement one mutation function with a selectable policy so the two behaviors sit side by side.

1. **Detect the unsafe case.** `unsafe = x.recipe is not None`. That single boolean is the whole risk assessment.
2. **Strict policy.** When `policy == 'raise'` and the op is unsafe, we `raise RuntimeError` — matching torch. The mutation never happens.
3. **Lenient policy.** When `policy == 'warn'` and the op is unsafe, we call `warnings.warn(...)` and *still perform* the mutation. The graph may now be corrupt, but the user was told.
4. **Safe case.** A leaf (`recipe is None`) is mutated under both policies with no fuss.

The demo records a warning with `warnings.catch_warnings`, confirms the lenient path both warned and mutated, and confirms the strict path refused.

In [ ]:
import numpy as np
import warnings
from dataclasses import dataclass

@dataclass
class Recipe:
    func: object
    args: tuple
    kwargs: dict
    parents: dict

class MiniTensor:
    def __init__(self, array, recipe=None):
        self.array = np.asarray(array, dtype=np.float64)
        self.recipe = recipe

def add_inplace(x, y, policy='raise'):
    unsafe = x.recipe is not None
    if unsafe:
        msg = 'in-place op on a Tensor with a recipe is unsafe'
        if policy == 'raise':
            raise RuntimeError(msg)
        warnings.warn(msg, RuntimeWarning)
    x.array += y.array
    return x

node = MiniTensor([1.0, 1.0], recipe=Recipe(np.add, (), {}, {}))
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    add_inplace(node, MiniTensor([2.0, 2.0]), policy='warn')
print('lenient warned:', len(w) == 1, '| mutated anyway:', node.array.tolist())

node2 = MiniTensor([1.0, 1.0], recipe=Recipe(np.add, (), {}, {}))
try:
    add_inplace(node2, MiniTensor([2.0, 2.0]), policy='raise')
except RuntimeError:
    print('strict refused | unchanged:', node2.array.tolist())